In [26]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [27]:
from taxonomy_generators import build_tools, build_registry


tools = build_tools()
registry = build_registry()

In [28]:
from query_taxonomy.features import FeatureGroup


registry[FeatureGroup.STRUCTURED_IDENTIFIERS][0]

In [29]:
from litellm import completion
import instructor

response = completion(
  model="anthropic/claude-haiku-4-5-20251001",
  messages=[{"role": "user", "content": "Hello, how are you?"}]
)
print(response.choices[0].message.content)

Hello! I'm doing well, thank you for asking. I'm here and ready to help with whatever you need. How are you doing today?


In [30]:
import json 

from taxonomy_generators import build_tools, catalog

TOOLS = build_tools(seed=0)
RUNNERS = { spec.name: spec.run for spec in TOOLS }

CLAUDE_TOOLS = [
    {"name": s.name, "description": s.description, "input_schema": s.input_schema}
    for s in TOOLS
]

SYSTEM = """You enrich search queries with taxonomy features.
Given a sentence and target feature names:
1. call generate_surface for each requested feature,
2. weave the surfaces into the sentence so it stays a plausible search query,
3. call verify on your candidate with span targets (min_count=1 per feature),
4. if verify fails, adjust the text and verify again.
End with ONLY the final enriched sentence, nothing else."""


In [31]:
import json

import instructor
from litellm import completion
from pydantic import BaseModel, Field


class EnrichedQuery(BaseModel):
    text: str = Field(description="The enriched sentence that passes verify")
    features_used: list[str] = Field(description="Feature names woven in")


class Enricher:
    def __init__(self, model="anthropic/claude-haiku-4-5-20251001"):
        self.model = model
        self.client = instructor.from_litellm(completion)

    def enrich(self, sentence: str, features: list[str]) -> EnrichedQuery:
        messages = [
            {
                "role": "system",
                "content": SYSTEM,
            },
            {
                "role": "user",
                "content": (
                    f"Sentence: {sentence}\n"
                    f"Features: {', '.join(features)}"
                ),
            },
        ]

        while True:
            response = completion(
                model=self.model,
                messages=messages,
                tools=CLAUDE_TOOLS,
                tool_choice="auto",
                max_tokens=1024,
            )

            message = response.choices[0].message
            messages.append(message.model_dump())

            tool_calls = getattr(message, "tool_calls", None)
            if not tool_calls:
                break

            for call in tool_calls:
                args = json.loads(call.function.arguments)

                result = RUNNERS[call.function.name](**args)

                messages.append(
                    {
                        "role": "tool",
                        "tool_call_id": call.id,
                        "content": json.dumps(result),
                    }
                )

        return self.client.chat.completions.create(
            model=self.model,
            response_model=EnrichedQuery,
            messages=messages
            + [
                {
                    "role": "user",
                    "content": "Return the final verified enrichment as structured output.",
                }
            ],
        )

In [32]:
print([entry["feature"] for entry in catalog()][:20])

['structured_identifiers:cve', 'structured_identifiers:datetime', 'structured_identifiers:cidr', 'structured_identifiers:ip_address', 'structured_identifiers:mac_address', 'structured_identifiers:uri', 'structured_identifiers:email', 'structured_identifiers:api_key', 'structured_identifiers:uuid', 'structured_identifiers:env_var', 'structured_identifiers:hex_color', 'structured_identifiers:standards_citation', 'structured_identifiers:currency_amount', 'structured_identifiers:iban', 'structured_identifiers:lei', 'structured_identifiers:industry_code', 'structured_identifiers:business_registration', 'structured_identifiers:legal_citation', 'structured_identifiers:court_docket', 'structured_identifiers:neutral_citation']


In [ ]:
enricher = Enricher()
enricher.enrich('How to a get to this website', ['ip_address', 'api_key'])

In [ ]:
enricher = Enricher()
enricher.enrich('Conforting home for 200 euro/month', ['datetime'])

EnrichedQuery(text='Conforting home for 200 euro/month since 1523759459', features_used=['datetime'])

In [ ]:
enricher = Enricher()
enricher.enrich('Blue is the color of month', ['hex_color'])

EnrichedQuery(text='Blue #cdE20D is the color of month', features_used=['hex_color'])

In [ ]:
from taxonomy_generators import SpanTarget, Targets, verify


features = ["ip_address", "api_key"]
result = enricher.enrich("How to a get to this website", features)

In [ ]:
report = verify(result.text, Targets(spans=tuple(SpanTarget(feature=f) for f in features)))

In [ ]:
print("passed: ", report.passed)

passed:  True


In [ ]:
for check in report.checks:
    print(f"  {check.target}: measured={check.measured}")

  ip_address: at least 1 span(s): measured=1.0
  api_key: at least 1 span(s): measured=1.0


## Augmentation part 

Lets check how the pseudo-code was implemented:
```python
while hungry_features_exist(order_sheet):
    feature = select_most_hungry(order_sheet)            # a span floor OR a stat band

    if is_stat(feature):                                 # e.g. length_words:60+
        band  = hungry_band(feature)                     # the thin part of the distribution
        query = find_query_movable_into(band)            # from the STATS table: short, zero-span
        new_query, ok = rewrite_towards(query, band)     # LLM expands / compresses / restructures
        answer = answer_of(query)                        # meaning kept -> answer unchanged

    elif needs_document(feature):                        # identifiers
        query, doc, surface = find_pair(feature)         # SPANS table: doc contains the surface
                                                         # + STATS table: query profile fits
        new_query, ok = enrich(query, surface)
        answer = doc

    else:                                                # markers, operator_syntax
        query = find_query(feature)                      # eligibility is often a STAT rule
        new_query, ok = enrich(query, feature)
        answer = answer_of(query)

    ok = ok and verify(new_query, feature)               # spans: feature present
                                                         # stats: scalar landed in the band
                                                         # both: no unwanted new spans
    if ok: write(new_query, answer)
```

### Setup the environment

In [1]:
import pandas as pd 
from augmentation import AugmentationLoop, Augmenter, GeneratedPool, operator_for
from composition import TargetComposition

selection = TargetComposition().build()

loop = AugmentationLoop(selection)

/Users/andrei/projects/hybrid-search-rrf-dataset/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# hungry_features
loop.hungry()

,floor,missing,operator,gate,runnable
0,id:datetime,997.00,inject,coherence_gate,False
1,id:medical,992.50,inject,coherence_gate,False
2,logical:operator_syntax,965.00,operator_syntax_rewrite,declaration_audit,False
3,id:logistics,890.50,inject,coherence_gate,False
4,id:media,875.75,inject,coherence_gate,False
5,marker:greeting,861.00,decorate,none,True
6,marker:interjection,853.00,decorate,none,True
7,id:network,433.75,inject,coherence_gate,False
8,length_words:60+,426.00,stat_rewrite,declaration_audit,False
9,id:shape_guess,135.75,inject,coherence_gate,False


In [3]:
loop.order_sheet()

,slice,floor,amount,credit,missing,reason
0,A,id:datetime,1000.0,3.00,997.00,exhausted
1,A,id:medical,1000.0,7.50,992.50,exhausted
2,A,logical:operator_syntax,1000.0,35.00,965.00,exhausted
3,A,id:logistics,1000.0,109.50,890.50,exhausted
4,A,id:media,1000.0,124.25,875.75,exhausted
5,A,marker:greeting,1000.0,139.00,861.00,exhausted
6,A,marker:interjection,1000.0,147.00,853.00,exhausted
7,A,id:network,1000.0,566.25,433.75,capped
8,C,length_words:60+,714.0,288.00,426.00,exhausted
9,A,id:shape_guess,1000.0,864.25,135.75,exhausted


In [4]:
declaration = operator_for('marker:politeness').declaration

print(declaration)

operator='decorate' floor_prefix='marker:' grounding=<Grounding.NONE: 'none'> answer_key=<AnswerKeyPath.INHERIT: 'inherit'> meaning_preserved=True verifiable_by='target marker span present on local re-measure' credit_gate=<CreditGate.NONE: 'none'> tool_loop=False


In [5]:
operator = operator_for("marker:acronym")

operator

In [6]:
operator = operator_for("id:datetime")

operator

In [7]:
row = selection.iloc[0]
row.query, list(row.floors)

('DHA', ['id:shape_guess', 'marker:acronym'])

In [8]:
op = operator_for("marker:politeness")
parents = op.eligible(selection, "marker:politeness")

len(parents)

32871

In [9]:
parent = parents.iloc[0]
parent

dataset                          beir-nfcorpus
query_id                            PLAIN-1018
checkable                                 True
slice                                        A
label_lane                               qrels
floors        [id:shape_guess, marker:acronym]
query                                      DHA
Name: 0, dtype: object

In [15]:
op.instruction("marker:politeness", parents)

"Rewrite the user's search query by weaving in a politeness phrase (e.g. 'please', 'could you kindly'). Vocabulary inspirations (adapt freely): 'thanks', 'thank you', 'can you'. Pick a phrasing that fits the query's tone and world, and vary it — never default to one stock opener; it may sit at the start, middle, or end. Keep every content word and the meaning unchanged. Add no other information: no names, numbers, dates, or identifiers. Check your text with the verify tool (span feature 'politeness'), then return the final query text."

In [ ]:
op.targets("marker:politeness")

Targets(spans=(SpanTarget(feature='politeness', min_count=1, max_count=None),), stats=())

In [10]:
engine = Augmenter()
out = engine.run(op.instruction("marker:politeness"), "Query: what does DHA do", op.targets("marker:politeness"))

out.text, out.accepted, out.attempts

TypeError: DecorateOperator.instruction() missing 1 required positional argument: 'parent'

In [ ]:
engine = Augmenter()
out = engine.run(op.instruction("marker:interjection"), "Query: I went to speak with my colleague and found out...", op.targets("marker:negation"))

out.text, out.accepted, out.attempts

('Oh, I went to speak with my colleague and found out he could not help...',
 True,
 1)

In [11]:
target = "marker:greeting"
op = operator_for(target)
parents = op.eligible(selection, target)

len(parents)

34130

In [12]:
engine = Augmenter()

out = engine.run(op.instruction(target, parents), "Query: My math question is really hard", op.targets(target))

out.text, out.accepted, out.attempts

('Hey, my math question is really hard', True, 1)

In [13]:
query= 'why can pregnant women be mean'

out = engine.run(op.instruction(target, parents), query, op.targets(target))

out.text, out.accepted, out.attempts

('hey, why can pregnant women be mean', True, 1)

In [14]:
query = 'Consider the rectangle with vertices at $(5,4),$ $(5,-4),$ $(-5,4),$ $('

out = engine.run(op.instruction(target, parents), query, op.targets(target))

out.text, out.accepted, out.attempts

('Howdy, consider the rectangle with vertices at $(5,4),$ $(5,-4),$ $(-5,4),$ and $(-5,-4)$.',
 True,
 1)

In [21]:
query = 'what important distinction did president john f. kennedy make regarding civil riights'

out = engine.run(op.instruction(target, parents), query, op.targets(target))

out.text, out.accepted, out.attempts

('hey, what important distinction did president john f. kennedy make regarding civil rights',
 True,
 1)

| feature (floor)     | hungry credits | status       |
| ------------------- | -------------: | ------------ |
| marker:politeness   |             14 | runnable now |
| marker:greeting     |            861 | runnable now |
| marker:interjection |            853 | runnable now |

In [ ]:
t = op.targets("marker:politeness")
engine.accept("could you please explain what DHA does", t).passed  == True # True
engine.accept("what does DHA do", t)                                # passed=False, checks name the miss

VerifyReport(passed=False, checks=(TargetCheck(target='politeness: at least 1 span(s)', measured=0.0, passed=False),))

In [ ]:
from taxonomy_generators.verify import SpanTarget, Targets

two_polite = Targets(spans=(SpanTarget(feature="politeness", min_count=2),))

out = engine.run(
    "Rewrite the user's search query by weaving in TWO distinct politeness "
    "phrases (e.g. 'please' and 'could you kindly'). Keep every content word "
    "and the meaning unchanged; add no other information. Check with the "
    "verify tool, then return the final query text.",
    "Query: what does DHA do",
    two_polite,
)

out

AugmentationOutcome(text='Please tell me what does DHA do, could you kindly?', accepted=True, attempts=1, checks=(TargetCheck(target='politeness: at least 2 span(s)', measured=3.0, passed=True),))

In [6]:
from composition import TargetComposition
from augmentation import AugmentationLoop

loop = AugmentationLoop(TargetComposition().build())
loop.hungry()                  

,floor,missing,operator,gate,runnable
0,id:datetime,997.00,inject,coherence_gate,False
1,id:medical,992.50,inject,coherence_gate,False
2,logical:operator_syntax,963.00,operator_syntax_rewrite,declaration_audit,False
3,id:logistics,886.50,inject,coherence_gate,False
4,id:media,871.25,inject,coherence_gate,False
5,id:network,428.25,inject,coherence_gate,False
6,length_words:60+,426.00,stat_rewrite,declaration_audit,False
7,id:shape_guess,76.75,inject,coherence_gate,False


In [ ]:
politeness_df = loop.run("marker:politeness")

In [20]:
import pandas as pd
pd.set_option("display.max_colwidth", None)   # stop pandas' own truncation

pool = pd.read_parquet("data/augmentation/pool.parquet")

# full before/after pairs: join the parent text back in
before = selection[["dataset", "query_id", "query"]].rename(columns={
    "dataset": "parent_dataset", "query_id": "generated_from", "query": "before",
})
pairs = pool.merge(before, on=["parent_dataset", "generated_from"])

for r in pairs.itertuples():
    print(f"--- {r.query_id}  ({r.floor}, {r.attempts} attempt(s))")
    print("before:", r.before)
    print("after: ", r.query)
    print()

--- aug-marker-politeness-4083  (marker:politeness, 1 attempt(s))
before: The only difference between easy and hard versions is the number of elements in the array.

You are given an array $a$ consisting of $n$ integers. In one move you can choose any $a_i$ and divide it by $2$ rounding down (in other words, in one move you can set $a_i := \lfloor\frac{a_i}{2}\rfloor$).

You can perform such an operation any (possibly, zero) number of times with any $a_i$.

Your task is to calculate the minimum possible number of operations required to obtain at least $k$ equal numbers in the array.

Don't forget that it is possible to have $a_i = 0$ after some operations, thus the answer always exists.
after:  Could you kindly help me understand this problem: The only difference between easy and hard versions is the number of elements in the array. You are given an array $a$ consisting of $n$ integers. In one move you can choose any $a_i$ and divide it by $2$ rounding down (in other words, in one move

In [16]:
loop.hungry()                  

,floor,missing,operator,gate,runnable
0,id:datetime,997.00,NaN,NaN,False
1,id:medical,992.50,NaN,NaN,False
2,logical:operator_syntax,965.00,NaN,NaN,False
3,id:logistics,890.50,NaN,NaN,False
4,id:media,875.75,NaN,NaN,False
5,marker:greeting,861.00,decorate,none,True
6,marker:interjection,853.00,decorate,none,True
7,id:network,433.75,NaN,NaN,False
8,length_words:60+,426.00,NaN,NaN,False
9,id:shape_guess,135.75,NaN,NaN,False


In [ ]:
loop.run("marker:greeting")

                                                                                                  
augment:marker:politeness:  36%|███▌      | 5/14 [02:03<00:41,  4.60s/row, attempted=5, dropped=0]

+ 984840 (1 attempt(s))
    before: 'why can pregnant women be mean'
    after:  'Hi there, why can pregnant women be mean'


                                                                                                  
augment:marker:politeness:  36%|███▌      | 5/14 [02:07<00:41,  4.60s/row, attempted=5, dropped=0]

+ MATH-q-4344 (1 attempt(s))
    before: 'Problem: Consider the rectangle with vertices at $(5,4),$ $(5,-4),$ $(-5,4),$ $('
    after:  'Hi there, consider the rectangle with vertices at $(5,4),$ $(5,-4),$ $(-5,4),$ $'


                                                                                                  
augment:marker:politeness:  36%|███▌      | 5/14 [02:12<00:41,  4.60s/row, attempted=5, dropped=0] 

...

                                                                                                  
augment:marker:politeness:  36%|███▌      | 5/14 [02:17<00:41,  4.60s/row, attempted=5, dropped=0] 

+ 1078330 (1 attempt(s))
    before: 'what important distinction did president john f. kennedy make regarding civil ri'
    after:  'Hi there, what important distinction did president john f. kennedy make regardin'


                                                                                                  
augment:marker:politeness:  36%|███▌      | 5/14 [02:21<00:41,  4.60s/row, attempted=5, dropped=0] 

+ 457076 (1 attempt(s))
    before: 'most common technology used in classroom'
    after:  'Hi there, most common technology used in classroom'


KeyboardInterrupt: 

In [6]:
floor = 'id:medical'

In [7]:
declaration = operator_for(floor).declaration

print(declaration)

operator='inject' floors='id:<domain> and id:<general-bank> identifier floors' grounding=<Grounding.DOC_COPIED: 'doc_copied'> answer_key=<AnswerKeyPath.MINTED: 'minted'> meaning_preserved=False verifiable_by='target bank span present + copied surface literally in the text + no other identifier floor gained (structural)' credit_gate=<CreditGate.COHERENCE_GATE: 'coherence_gate'> tool_loop=False


In [8]:

op = operator_for(floor)
parents = op.eligible(selection, floor)

len(parents)

82

In [10]:
engine = Augmenter()

In [12]:
query = 'what important distinction did president john f. kennedy make regarding civil riights'

out = engine.run(op.instruction(floor, parents), query, op.targets(floor, parents))

out.text, out.accepted, out.attempts

('what important distinction did president john f. kennedy make regarding civil rights 27    clinical_trial_id\n2     genomic_accession\n62    clinical_trial_id\n43    genomic_accession\n8     clinical_trial_id\n            ...        \n78    clinical_trial_id\n67         medical_code\n64    genomic_accession\n47    clinical_trial_id\n44    clinical_trial_id\nName: bank, Length: 82, dtype: str',
 False,
 2)

In [13]:
loop.run("id:medical", n=10)

NOTE: 'inject' is gated by coherence_gate (d42h) — rows are produced as feature-stock (no floor credit; skipped by the mini-fill) until the gate's pilot passes.


augment:id:medical:  10%|█         | 1/10 [00:00<00:08,  1.01row/s, attempted=1, dropped=0]

+ PLAIN-1741 (1 attempt(s))
    before: 'nuts'
    after:  'nuts health benefits NCT00123456 study results'


augment:id:medical:  20%|██        | 2/10 [00:02<00:08,  1.06s/row, attempted=2, dropped=0]

+ PLAIN-2640 (1 attempt(s))
    before: 'Chicken Salmonella Thanks to Meat Industry Lawsuit'
    after:  'Chicken Salmonella Thanks to Meat Industry Lawsuit 177C genomic accession'


...

augment:id:medical:  90%|█████████ | 9/10 [00:09<00:01,  1.14s/row, attempted=9, dropped=0]

+ PLAIN-924 (1 attempt(s))
    before: 'cocaine'
    after:  'cocaine NCT01516021 clinical trial'


augment:id:medical: 100%|██████████| 10/10 [00:10<00:00,  1.08s/row, attempted=10, dropped=0]

+ PLAIN-2780 (1 attempt(s))
    before: 'Do Fruit & Nut Bars Cause Weight Gain?'
    after:  'Do Fruit & Nut Bars Cause Weight Gain according to NCT01307904'
id:medical: accepted 10/10 attempts (need 10, parents available 82) -> data/augmentation/pool.parquet


,query_id,query,floor,operator,provenance,generated_from,parent_dataset,home_lane,grounding_doc_id,meaning_preserved,answer_key,attempts,credit_gate
0,aug-id-medical-PLAIN-1741,nuts health benefits NCT00123456 study results,id:medical,inject,augmented,PLAIN-1741,beir-nfcorpus,beir-nfcorpus,MED-1389,False,minted,1,coherence_gate
1,aug-id-medical-PLAIN-2640,Chicken Salmonella Thanks to Meat Industry Law...,id:medical,inject,augmented,PLAIN-2640,beir-nfcorpus,beir-nfcorpus,MED-3094,False,minted,1,coherence_gate
2,aug-id-medical-PLAIN-3432,Healthy Chocolate Milkshakes NCT01307904 recip...,id:medical,inject,augmented,PLAIN-3432,beir-nfcorpus,beir-nfcorpus,MED-3907,False,minted,1,coherence_gate
3,aug-id-medical-PLAIN-2470,Is Milk Good for Our Bones 177C,id:medical,inject,augmented,PLAIN-2470,beir-nfcorpus,beir-nfcorpus,MED-3094,False,minted,1,coherence_gate
4,aug-id-medical-PLAIN-2710,Artificial Food Colors and ADHD NCT01252433 cl...,id:medical,inject,augmented,PLAIN-2710,beir-nfcorpus,beir-nfcorpus,MED-3369,False,minted,1,coherence_gate
5,aug-id-medical-PLAIN-2396,Yale NCT00958308 clinical trials,id:medical,inject,augmented,PLAIN-2396,beir-nfcorpus,beir-nfcorpus,MED-3688,False,minted,1,coherence_gate
6,aug-id-medical-PLAIN-2197,sweeteners NCT01516021,id:medical,inject,augmented,PLAIN-2197,beir-nfcorpus,beir-nfcorpus,MED-3053,False,minted,1,coherence_gate
7,aug-id-medical-PLAIN-123,How Citrus Might Help Keep Your Hands Warm NCT...,id:medical,inject,augmented,PLAIN-123,beir-nfcorpus,beir-nfcorpus,MED-3477,False,minted,1,coherence_gate
8,aug-id-medical-PLAIN-924,cocaine NCT01516021 clinical trial,id:medical,inject,augmented,PLAIN-924,beir-nfcorpus,beir-nfcorpus,MED-3053,False,minted,1,coherence_gate
9,aug-id-medical-PLAIN-2780,Do Fruit & Nut Bars Cause Weight Gain accordin...,id:medical,inject,augmented,PLAIN-2780,beir-nfcorpus,beir-nfcorpus,MED-3897,False,minted,1,coherence_gate


### Run on all features

In [14]:
from augmentation.supply import SupplyIndex

SupplyIndex().build_all()

surfaces:bright-aops: 100%|██████████| 10000/10000 [00:08<00:00, 1158.81it/s]


bright-aops: 107,524 surfaces from 10,000 docs -> data/bright-aops/surfaces.parquet


surfaces:bright-leetcode: 100%|██████████| 10000/10000 [00:10<00:00, 989.11it/s]


bright-leetcode: 107,163 surfaces from 10,000 docs -> data/bright-leetcode/surfaces.parquet


surfaces:bright-theoremqa-questions: 100%|██████████| 10000/10000 [00:08<00:00, 1120.02it/s]


...

rarb-math: 256,915 surfaces from 31,595 docs -> data/rarb-math/surfaces.parquet


surfaces:scifact: 100%|██████████| 5183/5183 [00:07<00:00, 731.82it/s]


scifact: 46,889 surfaces from 5,183 docs -> data/scifact/surfaces.parquet


surfaces:trec-dl-2022: 100%|██████████| 30000/30000 [00:07<00:00, 4247.39it/s]

trec-dl-2022: 108,722 surfaces from 30,000 docs -> data/trec-dl-2022/surfaces.parquet


In [15]:
ro = SupplyIndex().readout(selection)

In [16]:
from IPython.display import Markdown, display

def show(df: pd.DataFrame) -> None:
    display(Markdown(df.to_markdown(index=False)))

In [17]:
show(ro.pivot_table(index="floor", columns="lane", values="direct_parents", fill_value=0))

|   beir-nfcorpus |   bright-aops |   bright-leetcode |   bright-theoremqa-questions |   crumb-clinical-trial |   crumb-code-retrieval |   crumb-legal-qa |   crumb-paper-retrieval |   crumb-set-operation-entity-retrieval |   crumb-stack-exchange |   crumb-theorem-retrieval |   crumb-tip-of-the-tongue |   limit |   msmarco-passage-dev |   rarb-code |   rarb-math |   trec-dl-2022 |
|----------------:|--------------:|------------------:|-----------------------------:|-----------------------:|-----------------------:|-----------------:|------------------------:|---------------------------------------:|-----------------------:|--------------------------:|--------------------------:|--------:|----------------------:|------------:|------------:|---------------:|
|               0 |             0 |                 1 |                            0 |                      2 |                    114 |                0 |                       0 |                                      1 |                      2 |                         1 |                       109 |       0 |                    13 |           0 |           0 |              0 |
|              58 |             8 |                 7 |                           14 |                    103 |                    533 |              192 |                      24 |                                     20 |                      5 |                         2 |                         1 |       0 |                   130 |           3 |         186 |              3 |
|              51 |             0 |                 3 |                            0 |                     79 |                   2650 |              126 |                       8 |                                     53 |                      4 |                         0 |                         3 |       0 |                    22 |           2 |          38 |              0 |
|              82 |            18 |                 0 |                            2 |                     98 |                     71 |              426 |                      21 |                                     17 |                      5 |                         0 |                         0 |       0 |                    29 |           0 |          11 |              1 |
|             132 |             8 |                 3 |                           23 |                    100 |                    563 |              545 |                      35 |                                    146 |                     20 |                         1 |                        28 |       0 |                   136 |           2 |          80 |              5 |
|             299 |            52 |                29 |                           60 |                     28 |                   1876 |             2436 |                      57 |                                    383 |                     51 |                        56 |                        77 |      32 |                  2153 |          15 |         383 |             14 |

In [ ]:
from augmentation.campaign import AugmentationCampaign

campaign = AugmentationCampaign(AugmentationLoop(TargetComposition().build()))
campaign.plan()    # read the bill first (~1,750 rows: two big marker floors + pilots)
campaign.run()

## Campaign Plan

**Summary**
- **Hungry floors:** 11
- **Target rows:** 1,958
- **Estimated minimum LLM calls:** ≥ 1,958

| Floor                     | Missing | Operator                  | Gate                | Action    | Target Rows |
| ---------------------------| --------:| ---------------------------| ---------------------| -----------| ------------:|
| `id:datetime`             | 997.00  | `inject`                  | `coherence_gate`    | `produce` | 30          |
| `id:medical`              | 992.50  | `inject`                  | `coherence_gate`    | `produce` | 20          |
| `logical:operator_syntax` | 965.00  | `operator_syntax_rewrite` | `declaration_audit` | `produce` | 30          |
| `id:logistics`            | 890.50  | `inject`                  | `coherence_gate`    | `produce` | 30          |
| `id:media`                | 875.75  | `inject`                  | `coherence_gate`    | `produce` | 30          |
| `marker:greeting`         | 861.00  | `decorate`                | `none`              | `produce` | 861         |
| `marker:interjection`     | 853.00  | `decorate`                | `none`              | `produce` | 853         |
| `id:network`              | 433.75  | `inject`                  | `coherence_gate`    | `produce` | 30          |
| `length_words:60+`        | 426.00  | `stat_rewrite`            | `declaration_audit` | `produce` | 30          |
| `id:shape_guess`          | 135.75  | `inject`                  | `coherence_gate`    | `produce` | 30          |
| `marker:politeness`       | 14.00   | `decorate`                | `none`              | `produce` | 14          |

In [ ]:
from augmentation import GeneratedPool
from composition.admission import MiniFill

# the only door into the composition: gated rows stay feature-stock, the rest
# compete for the hungry floors and the order sheet is re-emitted
pool = GeneratedPool().load()
admitted = MiniFill().admit(pool)

# born_for = the floor the row was generated FOR; floors = everything it
# credits after re-measurement (a greeting row can also carry politeness)
sel = pd.read_parquet(TargetComposition().selection_path)
children = sel[sel["generated_from"].notna()].merge(
    pool[["query_id", "floor", "operator"]].rename(columns={"floor": "born_for"}),
    on="query_id",
    how="left",
)
print(f"selection {len(sel):,} rows | augmented {len(children):,}")
print(children["born_for"].value_counts().to_string())

children[
    ["dataset", "query_id", "generated_from", "born_for", "operator", "floors", "query"]
].head(10)